In [1]:
import sys
sys.path.append('/home/lo276838/Modèles/fastmri_tf_vs_torch')

# import os.path as op
import time
import tensorflow as tf
import numpy as np

# from tensorflow.python.keras.callbacks import TensorBoard, ModelCheckpoint
# from tensorflow_addons.callbacks import TQDMProgressBar

# from autre.fastmri_tf.data.sequences.fastmri_sequences import ZeroFilled2DSequence
from autre.fastmri_tf.models.functional_models.unet import unet
# from fastmri_compare.vcr_C.vcr_tf import virtual_coil_reconstruction
from helpLena import virtual_coil_reconstruction

from autre.fastmri_tf.data.utils.h5 import from_multicoil_train_file_to_image_and_kspace
from autre.fastmri_tf.evaluate.reconstruction.zero_filled_reconstruction import zero_filled_cropped_recon
from autre.fastmri_tf.data.utils.masking.gen_mask import gen_mask


/volatile/Lena/Environments/envL/lib/python3.10/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
/volatile/Lena/Environments/envL/lib/python3.10/site-packages/tensorflow_addons/utils/ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.13.0 and strictly below 2.16.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.8.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file 

In [2]:
def filename_to_image_and_kspace(filename) :
    _ , kspace_multi = from_multicoil_train_file_to_image_and_kspace(filename)
    # kspace_multi = tf.convert_to_tensor(kspace_multi)
    image_multi = tf.signal.ifft2d(kspace_multi)
    image = virtual_coil_reconstruction(image_multi)
    kspace = tf.signal.fft2d(image)
    return image, kspace


train_path = "/volatile/FastMRI/brain_multicoil_train/multicoil_train/file_brain_AXT1POST_201_6002780.h5"

af = 4
lr = 1e-3

image , kspace = filename_to_image_and_kspace(train_path)
target = tf.abs(image)

2024-01-31 14:26:13.933261: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudnn.so.8'; dlerror: libcudnn.so.8: cannot open shared object file: No such file or directory
2024-01-31 14:26:13.933296: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1850] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2024-01-31 14:26:13.933679: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [47]:
zero_img_batch = list()

for batch in range (kspace.shape[0]) :
    mask = gen_mask(kspace[batch], accel_factor=af)
    masked_kspace = kspace[batch] * mask
    zero_filled = tf.signal.ifft2d(masked_kspace)
    zero_filled = tf.expand_dims(zero_filled,0)
    zero_img_batch.append(zero_filled)

zero_filled = tf.concat(zero_img_batch, axis = 0)
input = tf.abs(zero_filled)
print(input.shape)

(16, 640, 320)


In [48]:
run_params = {
    'n_layers': 4,
    'pool': 'max',
    "layers_n_channels": [16, 32, 64, 128],
    'layers_n_non_lins': 2,
}

model_unet = unet(input_size=input.shape, lr=lr, **run_params)

criterion = tf.keras.losses.MeanSquaredError()
optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

loss_list = []
num_epochs = 2 # 300

/volatile/Lena/Environments/envL/lib/python3.10/site-packages/tensorflow_addons/optimizers/rectified_adam.py:121: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


In [49]:
print(input.shape)
input = tf.expand_dims(input, 0)
print(input.shape)

(16, 640, 320)
(1, 16, 640, 320)


In [57]:
for epoch in range(num_epochs):
    
    with tf.GradientTape() as tape:
        outputs = model_unet(input)
        loss = criterion(outputs, target)

    loss_list.append(loss)

    gradients = tape.gradient(loss_list, model_unet.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model_unet.trainable_variables))

    print(f'Epoch [{epoch + 1}/{num_epochs}]')


Epoch [1/2]
Epoch [2/2]


In [58]:
print(outputs.shape)
print(loss)
print(loss_list)
print(gradients)

(1, 16, 640, 320)
tf.Tensor(4.5028594e-13, shape=(), dtype=float32)
[<tf.Tensor: shape=(), dtype=float32, numpy=4.1843915e-13>, <tf.Tensor: shape=(), dtype=float32, numpy=7.402353e-13>, <tf.Tensor: shape=(), dtype=float32, numpy=6.567828e-13>, <tf.Tensor: shape=(), dtype=float32, numpy=4.597441e-13>, <tf.Tensor: shape=(), dtype=float32, numpy=4.3920076e-13>, <tf.Tensor: shape=(), dtype=float32, numpy=4.5028594e-13>]
[<tf.Tensor: shape=(3, 3, 320, 16), dtype=float32, numpy=
array([[[[-9.06330320e-19, -1.78350851e-20,  1.82256034e-18, ...,
          -7.70237396e-19, -1.26547901e-18,  2.43661721e-20],
         [-8.92818118e-19, -1.91942899e-20,  1.76309867e-18, ...,
          -7.40986428e-19, -1.20893470e-18,  2.03755329e-20],
         [-8.30271787e-19, -1.89351029e-20,  1.66296452e-18, ...,
          -7.21269286e-19, -1.17749181e-18,  2.02709753e-20],
         ...,
         [-9.71730425e-19, -1.79295857e-20,  1.72935114e-18, ...,
          -8.74782065e-19, -1.47516888e-18,  2.50902378e-2